# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aniqaatiq842-commits/Flyrank-ML-INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import getpass

HF_TOKEN = getpass.getpass("Paste your Hugging Face token here: ")

print("Token loaded successfully:", bool(HF_TOKEN))

Paste your Hugging Face token here: ··········
Token loaded successfully: True


In [6]:
!pip install -q duckdb huggingface_hub

In [7]:
import duckdb

con = duckdb.connect()

print("DuckDB connected!")

DuckDB connected!


In [8]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

print(FACT)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance


In [9]:
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [10]:
schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
    LIMIT 1
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
DIM_CONTENT = f"{REL}/dim_content.parquet"

content_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DIM_CONTENT}')
    LIMIT 1
""").df()

content_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


## 1. Unit of analysis + time window

**One row** = One row represents the daily performance record for one content item for one client on one reporting date
I will use fact_content_daily_performance as the main performance table and dim_content for content-level metadata. The fact table provides daily Google Search Console and GA4 performance, while dim_content provides attributes such as content type, word count, backlinks, and publication status.
##What time window?
will use February 2026 as the feature window and decision-time history. March 2026 will be used only as the future outcome window for constructing the decline proxy.
##Prediction / ranking target
I will rank content items by whether their organic Google Search Console clicks decline from February to March 2026. The proxy label is 1 when March organic clicks are lower than February organic clicks, and 0 otherwise.
### Deliberate exclusion

I deliberately exclude March performance variables from the feature set because March occurs after the decision point. Including March information would leak the future outcome into the model.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## 2) Decision and label definition

### Decision moment

The decision is made at the end of February 2026. At this point, only information available through February 2026 can be used to rank content for a potential refresh.

### Feature window

February 2026. Features will be aggregated from daily performance observations during this month.

### Outcome window

March 2026. March is reserved for measuring the subsequent change in organic performance and is not used to construct the predictive features.

### Label/proxy

A content item is labeled as declining (`is_declining_label = 1`) when its total GSC clicks in March 2026 are lower than its total GSC clicks in February 2026. Otherwise, the label is 0. This is a proxy for declining organic performance rather than a direct claim that the content definitely needs a refresh.

### Why this proxy

GSC clicks directly measure organic search traffic reaching the content, so a month-over-month decline provides a simple, observable signal for prioritizing content for further review.

### Availability rule

GSC performance is used only for rows where `gsc_data_available IS TRUE`, so unavailable GSC observations are not treated as zero.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain

I expect the fact table to have one record per client, content item, and reporting date. I check whether the combination of `client_hash_id`, `content_hash_id`, and `report_date` uniquely identifies rows in the March 2026 slice.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS distinct_client_content_dates,
    COUNT(*) -
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS duplicate_rows
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""").df()

q1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_client_content_dates,duplicate_rows
0,9841378,9841378,0


In [13]:
### Query 2 — March 2026 row count and date span

### I check the March 2026 slice to verify its number of rows and the earliest and latest reporting dates present.
q2 = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""").df()

q2

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


**Result:** The March 2026 slice contains 9,841,378 rows, with reporting dates ranging from 2026-03-01 to 2026-03-31. This confirms that the selected slice covers the complete March 2026 reporting window.

In [14]:
### Query 3 — GSC data availability

### I check how many March 2026 rows have usable Google Search Console data. I use `IS TRUE` explicitly so that only rows marked as having available GSC data are counted as available.
q3 = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS gsc_unavailable_rows
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
""").df()

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,gsc_unavailable_rows
0,9841378,3611061,6230317


**Result:** Of the 9,841,378 rows in the March 2026 slice, 3,611,061 rows have GSC data available and 6,230,317 rows do not. The availability filter uses `gsc_data_available IS TRUE`, so only rows explicitly marked as having GSC data available are treated as usable for GSC-based analysis.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Five-feature frame

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_df = con.execute(f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feb_gsc_impressions,
        SUM(gsc_clicks) AS feb_gsc_clicks,
        AVG(gsc_avg_position) AS feb_gsc_avg_position,
        SUM(sessions_organic) AS feb_organic_sessions,
        SUM(ga4_total_engagement_sec) AS feb_ga4_engagement_sec

    FROM read_parquet('{FACT}/month=2026-02/*.parquet')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM february
""").df()

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_organic_sessions,feb_ga4_engagement_sec
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,0.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,0.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,193.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,0.0


### Future outcome / proxy label

The label is constructed after the decision point using March 2026 GSC clicks. A content item receives `is_declining_label = 1` when its March GSC clicks are lower than its February GSC clicks, and 0 otherwise.

In [16]:
label_df = con.execute(f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks
    FROM read_parquet('{FACT}/month=2026-02/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS mar_clicks
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.feb_clicks,
    m.mar_clicks,

    CASE
        WHEN m.mar_clicks < f.feb_clicks THEN 1
        ELSE 0
    END AS is_declining_label

FROM february f
INNER JOIN march m
    USING (client_hash_id, content_hash_id)
""").df()

label_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,feb_clicks,mar_clicks,is_declining_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,7.0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.0,0.0,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,4.0,6.0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,19.0,13.0,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,0.0,1.0,0


In [17]:
model_df = feature_df.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Rows in final modeling frame:", len(model_df))
model_df.head()

Rows in final modeling frame: 134238


,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_organic_sessions,feb_ga4_engagement_sec,is_declining_label
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0,0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,0.0,1
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,0.0,0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,193.0,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,0.0,1


In [18]:
model_df["is_declining_label"].value_counts(dropna=False)

,count
is_declining_label,
0,109781
1,24457


In [19]:
### Deliberate leakage experiment

##To demonstrate target leakage, I intentionally add a feature derived directly from the future label. This feature would never be available at the decision moment. I expect the quick model score to become unrealistically high, demonstrating why label-derived information must be excluded.
model_df["leaked_label_feature"] = model_df["is_declining_label"]

In [21]:
HONEST_FEATURES = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_organic_sessions",
    "feb_ga4_engagement_sec"
]

X = model_df[HONEST_FEATURES]
y = model_df["is_declining_label"]

In [23]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

HONEST_FEATURES = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_organic_sessions",
    "feb_ga4_engagement_sec"
]

X = model_df[HONEST_FEATURES].copy()
y = model_df["is_declining_label"].copy()

# Replace infinite values with missing values
X = X.replace([np.inf, -np.inf], np.nan)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Impute missing values + scale + model
honest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_predictions
)

print("Honest accuracy:", honest_accuracy)

Honest accuracy: 0.8169323599523242


In [24]:
model_df["leaked_label_feature"] = model_df["is_declining_label"]

In [25]:
LEAKED_FEATURES = HONEST_FEATURES + [
    "leaked_label_feature"
]

X_leak = model_df[LEAKED_FEATURES].copy()
y = model_df["is_declining_label"].copy()

X_leak = X_leak.replace([np.inf, -np.inf], np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leak_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

leak_model.fit(X_train, y_train)

leak_predictions = leak_model.predict(X_test)

leak_accuracy = accuracy_score(
    y_test,
    leak_predictions
)

print("Accuracy WITH leakage:", leak_accuracy)

Accuracy WITH leakage: 0.9999627532777116


In [26]:
model_df.drop(
    columns=["leaked_label_feature"],
    inplace=True
)

### Leakage lesson

I intentionally added `leaked_label_feature`, which was directly derived from the target label. This feature represents information that would not be available at the February decision moment. As expected, including it produced an unrealistically high score.

After removing the leaked feature, I retrained the model using only the five features available at the decision moment. The resulting honest score is the number that should be retained as the baseline.

This demonstrates why features must be constructed only from information that would genuinely have been available when the decision was made.

In [27]:
print(model_df.columns.tolist())

['client_hash_id', 'content_hash_id', 'feb_gsc_impressions', 'feb_gsc_clicks', 'feb_gsc_avg_position', 'feb_organic_sessions', 'feb_ga4_engagement_sec', 'is_declining_label']


### 4) Limitation

A key limitation of this slice is GSC data availability. In the March 2026 verification slice, 3,611,061 of 9,841,378 rows have GSC data available. Therefore, GSC-based analysis does not cover every daily client-content record. Missing GSC availability should not be interpreted as zero search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.